# Cluster hybrid — predict with **live OneMap geo** (8 inputs)

Companion to `03_hybrid_prediction.ipynb`. Same eight inputs, same hybrid bundle, **same starting feature vector** built via the feature-table lookup — but then the geo columns (`dist_to_mrt_m`, `dist_to_foodcourt_m`, `dist_to_nearest_mall_m`, `mall_count_3km`, `mall_weighted_access_3km`, `dist_to_nearest_school_m`, `school_count_1km`, `primary_school_count_1km`, `dist_to_highway_m`) are **recomputed live** from OneMap geocode + amenity CSVs (mirrors the override block in `backend/predict.py`).

Use this to compare against `03` and see how much the live-geo override moves the prediction. If `03b` consistently prices higher than `03` for the same listings, that's the same drift currently affecting the API.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import importlib
import sys
from pathlib import Path
import os

# Run from 03_ml_layer_hybrid/ or repo root
if not Path("yc_hybrid_inference.py").exists() and Path("exploration-on-yc-data/yc_hybrid_inference.py").exists():
    os.chdir("exploration-on-yc-data")

# Reload so Jupyter picks up edits to yc_hybrid_inference.py
import yc_hybrid_inference
importlib.reload(yc_hybrid_inference)

from yc_hybrid_inference import (
    load_bundle,
    predict_from_user_input,
    build_yc_hybrid_vector,
    predict_price,
    USER_INPUT_KEYS,
    default_feature_table_csv,
)

# Add SystemCode/ to sys.path so we can import backend.location
_HERE = Path.cwd().resolve()
_SYSTEM_CODE = None
for cand in [_HERE, *_HERE.parents]:
    if (cand / "backend" / "location.py").exists():
        _SYSTEM_CODE = cand
        break
if _SYSTEM_CODE is None:
    raise RuntimeError("Could not locate SystemCode/ root (looking for backend/location.py)")
if str(_SYSTEM_CODE) not in sys.path:
    sys.path.insert(0, str(_SYSTEM_CODE))

from backend.location import compute_nearby, nearest_highway_dist_m, _load_amenities  # noqa: E402
_load_amenities()

bundle = load_bundle()
print("Loaded bundle:", bundle["n_clusters"], "clusters")
print("Feature lookup table:", default_feature_table_csv().name)
print("SystemCode root:", _SYSTEM_CODE)

  Amenities loaded: mrt (161 records)
  Amenities loaded: hawker (127 records)
  Amenities loaded: mall (148 records)
  Amenities loaded: highway (328 records)
  Amenities loaded: school (179 records) from school_popularity_combined.csv
Loaded bundle: 3 clusters
Feature lookup table: hdb_feature_table_20260412.csv
SystemCode root: /Users/bhuvesh/Documents/PropertyLens/SystemCode


## Live-geo helpers

Replicates `_geocode_for_predict` and `_live_geo_features` from `backend/predict.py`. Geocode uses the OneMap public Search API (no key, ~8s timeout, three query variants). Live geo features are computed from the bundled amenity CSVs at the geocoded lat/lng.

Mall weighting uses the same big-mall regex (`MEGA|HUB|CITY|JUNCTION|POINT|PLAZA|CENTRE`) and the same `1.5 / (km + 0.25)` formula as `predict.py`.

In [2]:
import re
import requests
import numpy as np

_ONEMAP_SEARCH_URL = "https://www.onemap.gov.sg/api/common/elastic/search"
_BIG_MALL_RE = re.compile(r"MEGA|HUB|CITY|JUNCTION|POINT|PLAZA|CENTRE", re.I)


def geocode_block_street(block: str, street_name: str) -> tuple[float, float] | None:
    """Mirror of backend.predict._geocode_for_predict."""
    candidates = [
        f"{block} {street_name}",
        f"BLK {block} {street_name}",
        street_name,
    ]
    for q in candidates:
        try:
            resp = requests.get(
                _ONEMAP_SEARCH_URL,
                params={"searchVal": q, "returnGeom": "Y", "getAddrDetails": "Y", "pageNum": 1},
                timeout=8,
            )
            results = resp.json().get("results", [])
            if not results:
                continue
            for r in results:
                if str(r.get("BLK_NO", "")).strip() == str(block).strip():
                    lat = float(r.get("LATITUDE", 0))
                    lng = float(r.get("LONGITUDE", 0))
                    if lat or lng:
                        return lat, lng
            lat = float(results[0].get("LATITUDE", 0))
            lng = float(results[0].get("LONGITUDE", 0))
            if lat or lng:
                return lat, lng
        except Exception:
            continue
    return None


def live_geo_features(lat: float, lng: float) -> dict[str, float]:
    """Mirror of backend.predict._live_geo_features."""
    nearby = compute_nearby(lat, lng, radius_m=15000.0)
    feats: dict[str, float] = {}

    mrt_items = nearby.get("mrt", [])
    if mrt_items:
        feats["dist_to_mrt_m"] = float(mrt_items[0]["dist_m"])

    hawker_items = nearby.get("hawker", [])
    if hawker_items:
        feats["dist_to_foodcourt_m"] = float(hawker_items[0]["dist_m"])

    mall_items = nearby.get("mall", [])
    if mall_items:
        feats["dist_to_nearest_mall_m"] = float(mall_items[0]["dist_m"])

    malls_3km = [m for m in mall_items if m["dist_m"] <= 3000]
    feats["mall_count_3km"] = float(len(malls_3km))
    if malls_3km:
        wa = sum(
            (1.5 if _BIG_MALL_RE.search(m.get("name", "")) else 1.0)
            / (max(m["dist_m"] / 1000.0, 0.05) + 0.25)
            for m in malls_3km
        )
        feats["mall_weighted_access_3km"] = wa
    else:
        feats["mall_weighted_access_3km"] = 0.0

    school_items = nearby.get("school", [])
    if school_items:
        feats["dist_to_nearest_school_m"] = float(school_items[0]["dist_m"])

    schools_1km = [s for s in school_items if s["dist_m"] <= 1000]
    feats["school_count_1km"] = float(len(schools_1km))
    feats["primary_school_count_1km"] = float(len(schools_1km))

    hw_dist = nearest_highway_dist_m(lat, lng)
    if hw_dist is not None:
        feats["dist_to_highway_m"] = float(hw_dist)

    return feats


def predict_with_live_geo(user_input: dict, bundle) -> dict:
    """Build vector via feature-table lookup, then override geo columns from live OneMap.

    Returns the same keys as predict_from_user_input plus:
      - ``geocoded_latlng``: (lat, lng) or None
      - ``geo_overrides``: dict of {feature: live_value} that were applied
      - ``geo_deltas``: dict of {feature: (table_value, live_value)} for inspection
    """
    base = predict_from_user_input(**user_input, bundle=bundle)
    X = base["vector"].ravel().copy()
    feat_cols = bundle["feature_columns"]
    col_index = {c: i for i, c in enumerate(feat_cols)}

    coords = geocode_block_street(str(user_input["block"]).strip(), str(user_input["street_name"]).strip())
    overrides: dict[str, float] = {}
    deltas: dict[str, tuple[float, float]] = {}
    if coords is not None:
        lat, lng = coords
        live = live_geo_features(lat, lng)
        for name, val in live.items():
            if name in col_index:
                idx = col_index[name]
                deltas[name] = (float(X[idx]), float(val))
                X[idx] = val
                overrides[name] = float(val)

    pred = float(predict_price(X.reshape(1, -1), bundle))

    out = dict(base)
    out["vector"] = X.reshape(1, -1)
    out["predicted_resale_price"] = pred
    out["geocoded_latlng"] = coords
    out["geo_overrides"] = overrides
    out["geo_deltas"] = deltas
    return out

## Single listing — feature-table vs live-geo, side by side

Same Bishan listing as notebook `03` (Blk 163 Bishan St 13, listing 500073723, asking S$950,000). Edit `user_input` below for any other listing.

In [3]:
import pandas as pd
import numpy as np

actual_price = 950_000  # asking; set None to skip eval table

user_input = {
    "block": "163",
    "street_name": "BISHAN STREET 13",
    "town": "BISHAN",
    "flat_type": "5 ROOM",
    "floor_area_sqm": round(1302 * 0.09290304, 2),
    "storey_range": "13 TO 15",
    "lease_commence_date": 1987,
    "sale_month": "2026-04",
}

table_only = predict_from_user_input(**user_input, bundle=bundle)
live = predict_with_live_geo(user_input, bundle=bundle)

p_table = float(table_only["predicted_resale_price"])
p_live = float(live["predicted_resale_price"])

print(f"Feature-table prediction (03):     S${p_table:>12,.2f}")
print(f"Live-geo prediction      (03b):    S${p_live:>12,.2f}")
print(f"Live − table delta:                S${p_live - p_table:>+12,.2f}  ({(p_live - p_table)/max(p_table,1)*100:+.2f}%)")
print(f"Geocoded:                          {live['geocoded_latlng']}")
print(f"Lookup matched:                    {table_only['lookup_matched']} | {table_only['matched_address_key']}")
if table_only["imputation_note"]:
    print(f"Note: {table_only['imputation_note']}")

if live["geo_deltas"]:
    delta_df = pd.DataFrame(
        [(k, v[0], v[1], v[1] - v[0]) for k, v in live["geo_deltas"].items()],
        columns=["feature", "table_value", "live_value", "delta"],
    )
    print("\n--- Geo feature overrides (table → live) ---")
    print(delta_df.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
else:
    print("\n(no geo overrides applied — geocode failed or no overlap with feature columns)")

if actual_price is not None:
    rows = []
    for label, pred in [("03 (table)", p_table), ("03b (live)", p_live)]:
        err = pred - actual_price
        rows.append(
            {
                "variant": label,
                "actual": f"{actual_price:,.0f}",
                "predicted": f"{pred:,.2f}",
                "abs_error": f"{abs(err):,.2f}",
                "mape_pct": f"{abs(err)/actual_price*100:.4f}",
                "bias_pct": f"{err/actual_price*100:+.4f}",
            }
        )
    print("\n--- Evaluation vs actual_price ---")
    print(pd.DataFrame(rows).to_string(index=False))

Feature-table prediction (03):     S$  997,465.78
Live-geo prediction      (03b):    S$  982,244.81
Live − table delta:                S$  -15,220.97  (-1.53%)
Geocoded:                          (1.348022951695617, 103.8562749810033)
Lookup matched:                    True | 163 BISHAN ST 13

--- Geo feature overrides (table → live) ---
                 feature  table_value  live_value   delta
           dist_to_mrt_m     1,347.29      767.00 -580.29
     dist_to_foodcourt_m       888.01      882.00   -6.01
  dist_to_nearest_mall_m     1,263.54      879.00 -384.54
          mall_count_3km        13.00       15.00    2.00
mall_weighted_access_3km         7.20        8.06    0.86
dist_to_nearest_school_m       207.12      207.00   -0.12
        school_count_1km         6.00        3.00   -3.00
primary_school_count_1km         3.00        3.00    0.00
       dist_to_highway_m       468.83      944.00  475.17

--- Evaluation vs actual_price ---
   variant  actual  predicted abs_error mape_

/var/folders/t4/jqyfdvcn0xn_qjk47656g9840000gn/T/ipykernel_64113/3800571513.py:111: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pred = float(predict_price(X.reshape(1, -1), bundle))


## Bulk comparison — same listings as `03`

Runs both predictors on the same 24 PropertyGuru-style listings and prints them side by side. Look at:
- **`bias_table_pct` vs `bias_live_pct`** — does live-geo systematically push positive (overestimate)?
- **`live_minus_table_pct`** — per-row drift introduced by the override.
- **Aggregate row at the bottom** — MAE / RMSE / mean bias for each variant.

In [4]:
def sqft_to_sqm(sqft: float) -> float:
    return round(sqft * 0.09290304, 2)

SALE_MONTH = "2026-04"
STOREY = "07 TO 09"

listings = [
    {"listing_id": "pg-101-jurong-east-s13", "block": "101", "street_name": "JURONG EAST STREET 13", "town": "JURONG EAST", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(732), "storey_range": STOREY, "lease_commence_date": 1982, "sale_month": SALE_MONTH, "actual_price": 530_000},
    {"listing_id": "pg-244-jurong-east-s24", "block": "244", "street_name": "JURONG EAST STREET 24", "town": "JURONG EAST", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(764), "storey_range": STOREY, "lease_commence_date": 1982, "sale_month": SALE_MONTH, "actual_price": 380_000},
    {"listing_id": "pg-93-paya-lebar", "block": "93", "street_name": "PAYA LEBAR WAY", "town": "GEYLANG", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(624), "storey_range": STOREY, "lease_commence_date": 1972, "sale_month": SALE_MONTH, "actual_price": 300_000},
    {"listing_id": "pg-241-jurong-east-s24", "block": "241", "street_name": "JURONG EAST STREET 24", "town": "JURONG EAST", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(732), "storey_range": STOREY, "lease_commence_date": 1982, "sale_month": SALE_MONTH, "actual_price": 468_000},
    {"listing_id": "pg-311c-clementi-av4", "block": "311C", "street_name": "CLEMENTI AVENUE 4", "town": "CLEMENTI", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(646), "storey_range": STOREY, "lease_commence_date": 2015, "sale_month": SALE_MONTH, "actual_price": 675_000},
    {"listing_id": "pg-211-jurong-east-s21", "block": "211", "street_name": "JURONG EAST STREET 21", "town": "JURONG EAST", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(721), "storey_range": STOREY, "lease_commence_date": 1982, "sale_month": SALE_MONTH, "actual_price": 400_000},
    {"listing_id": "pg-131-cashew", "block": "131", "street_name": "CASHEW ROAD", "town": "BUKIT PANJANG", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(786), "storey_range": STOREY, "lease_commence_date": 1987, "sale_month": SALE_MONTH, "actual_price": 500_000},
    {"listing_id": "pg-18-bedok-south", "block": "18", "street_name": "BEDOK SOUTH ROAD", "town": "BEDOK", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(786), "storey_range": STOREY, "lease_commence_date": 1975, "sale_month": SALE_MONTH, "actual_price": 450_000},
    {"listing_id": "pg-235-bukit-batok-e5", "block": "235", "street_name": "BUKIT BATOK EAST AVENUE 5", "town": "BUKIT BATOK", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(743), "storey_range": STOREY, "lease_commence_date": 1984, "sale_month": SALE_MONTH, "actual_price": 425_000},
    {"listing_id": "pg-81-commonwealth", "block": "81", "street_name": "COMMONWEALTH CLOSE", "town": "QUEENSTOWN", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(614), "storey_range": STOREY, "lease_commence_date": 1964, "sale_month": SALE_MONTH, "actual_price": 350_000},
    {"listing_id": "pg-702-west-coast", "block": "702", "street_name": "WEST COAST ROAD", "town": "CLEMENTI", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(721), "storey_range": STOREY, "lease_commence_date": 1979, "sale_month": SALE_MONTH, "actual_price": 368_000},
    {"listing_id": "pg-643-amk-av5", "block": "643", "street_name": "ANG MO KIO AVENUE 5", "town": "ANG MO KIO", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(807), "storey_range": STOREY, "lease_commence_date": 1980, "sale_month": SALE_MONTH, "actual_price": 440_000},
    {"listing_id": "pg-24-hougang-av3", "block": "24", "street_name": "HOUGANG AVENUE 3", "town": "HOUGANG", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(721), "storey_range": STOREY, "lease_commence_date": 1977, "sale_month": SALE_MONTH, "actual_price": 400_000},
    {"listing_id": "pg-93-whampoa", "block": "93", "street_name": "WHAMPOA DRIVE", "town": "KALLANG/WHAMPOA", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(700), "storey_range": STOREY, "lease_commence_date": 1973, "sale_month": SALE_MONTH, "actual_price": 399_000},
    {"listing_id": "pg-629-hougang-av8", "block": "629", "street_name": "HOUGANG AVENUE 8", "town": "HOUGANG", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(689), "storey_range": STOREY, "lease_commence_date": 1986, "sale_month": SALE_MONTH, "actual_price": 430_000},
    {"listing_id": "pg-504-amk-av8", "block": "504", "street_name": "ANG MO KIO AVENUE 8", "town": "ANG MO KIO", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(731), "storey_range": STOREY, "lease_commence_date": 1979, "sale_month": SALE_MONTH, "actual_price": 488_888},
    {"listing_id": "pg-3-st-george", "block": "3", "street_name": "SAINT GEORGE'S ROAD", "town": "KALLANG/WHAMPOA", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(635), "storey_range": STOREY, "lease_commence_date": 1975, "sale_month": SALE_MONTH, "actual_price": 400_000},
    {"listing_id": "pg-333-amk-av1", "block": "333", "street_name": "ANG MO KIO AVENUE 1", "town": "ANG MO KIO", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(721), "storey_range": STOREY, "lease_commence_date": 1980, "sale_month": SALE_MONTH, "actual_price": 429_000},
    {"listing_id": "pg-5-ghim-moh", "block": "5", "street_name": "GHIM MOH ROAD", "town": "QUEENSTOWN", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(700), "storey_range": STOREY, "lease_commence_date": 1975, "sale_month": SALE_MONTH, "actual_price": 428_000},
    {"listing_id": "pg-104-potong-pasir", "block": "104", "street_name": "POTONG PASIR AVENUE 1", "town": "TOA PAYOH", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(797), "storey_range": STOREY, "lease_commence_date": 1984, "sale_month": SALE_MONTH, "actual_price": 618_000},
    {"listing_id": "pg-52-teban-gardens", "block": "52", "street_name": "TEBAN GARDENS ROAD", "town": "JURONG EAST", "flat_type": "4 ROOM", "floor_area_sqm": sqft_to_sqm(893), "storey_range": STOREY, "lease_commence_date": 1985, "sale_month": SALE_MONTH, "actual_price": 479_999},
    {"listing_id": "pg-18c-circuit", "block": "18C", "street_name": "CIRCUIT ROAD", "town": "GEYLANG", "flat_type": "4 ROOM", "floor_area_sqm": sqft_to_sqm(1001), "storey_range": STOREY, "lease_commence_date": 2015, "sale_month": SALE_MONTH, "actual_price": 979_000},
    {"listing_id": "pg-14-dover-close-east", "block": "14", "street_name": "DOVER CLOSE EAST", "town": "QUEENSTOWN", "flat_type": "5 ROOM", "floor_area_sqm": sqft_to_sqm(1281), "storey_range": STOREY, "lease_commence_date": 1977, "sale_month": SALE_MONTH, "actual_price": 938_000},
    {"listing_id": "pg-1-lorong-lew-lian", "block": "1", "street_name": "LORONG LEW LIAN", "town": "SERANGOON", "flat_type": "3 ROOM", "floor_area_sqm": sqft_to_sqm(689), "storey_range": STOREY, "lease_commence_date": 1978, "sale_month": SALE_MONTH, "actual_price": 430_000},
]

rows = []
for L in listings:
    actual = L.pop("actual_price")
    listing_id = L.pop("listing_id")
    table_pred = float(predict_from_user_input(**L, bundle=bundle)["predicted_resale_price"])
    live_res = predict_with_live_geo(L, bundle=bundle)
    live_pred = float(live_res["predicted_resale_price"])
    rows.append(
        {
            "listing_id": listing_id,
            "address": f"{L['block']} {L['street_name']}",
            "actual": actual,
            "pred_table": table_pred,
            "pred_live": live_pred,
            "bias_table_pct": (table_pred - actual) / actual * 100,
            "bias_live_pct": (live_pred - actual) / actual * 100,
            "live_minus_table": live_pred - table_pred,
            "live_minus_table_pct": (live_pred - table_pred) / max(table_pred, 1) * 100,
            "geocoded": live_res["geocoded_latlng"] is not None,
        }
    )

comp_df = pd.DataFrame(rows)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
pd.set_option("display.max_rows", 60)

disp = comp_df.copy()
for col in ["actual", "pred_table", "pred_live", "live_minus_table"]:
    disp[col] = disp[col].map(lambda x: f"{x:,.0f}")
for col in ["bias_table_pct", "bias_live_pct", "live_minus_table_pct"]:
    disp[col] = disp[col].map(lambda x: f"{x:+.2f}")

print("--- Per-listing: feature-table vs live-geo ---")
print(disp.to_string(index=False))

def _agg(label, preds, actuals):
    err = preds - actuals
    return {
        "variant": label,
        "n": len(preds),
        "MAE_SGD": float(np.mean(np.abs(err))),
        "RMSE_SGD": float(np.sqrt(np.mean(err ** 2))),
        "MAPE_pct": float(np.mean(np.abs(err / actuals)) * 100),
        "mean_bias_pct": float(np.mean(err / actuals) * 100),
    }

actuals = comp_df["actual"].to_numpy(dtype=float)
agg = pd.DataFrame(
    [
        _agg("03  (table)", comp_df["pred_table"].to_numpy(dtype=float), actuals),
        _agg("03b (live) ", comp_df["pred_live"].to_numpy(dtype=float), actuals),
    ]
)
print("\n--- Aggregate over all rows ---")
print(agg.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))
print(
    f"\nGeocode success rate: {comp_df['geocoded'].sum()}/{len(comp_df)}"
)

/var/folders/t4/jqyfdvcn0xn_qjk47656g9840000gn/T/ipykernel_64113/3800571513.py:111: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pred = float(predict_price(X.reshape(1, -1), bundle))
/var/folders/t4/jqyfdvcn0xn_qjk47656g9840000gn/T/ipykernel_64113/3800571513.py:111: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pred = float(predict_price(X.reshape(1, -1), bundle))
/var/folders/t4/jqyfdvcn0xn_qjk47656g9840000gn/T/ipykernel_64113/3800571513.py:111: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation

--- Per-listing: feature-table vs live-geo ---
            listing_id                       address  actual pred_table pred_live bias_table_pct bias_live_pct live_minus_table live_minus_table_pct  geocoded
pg-101-jurong-east-s13     101 JURONG EAST STREET 13 530,000    414,971   426,607         -21.70        -19.51           11,636                +2.80      True
pg-244-jurong-east-s24     244 JURONG EAST STREET 24 380,000    415,248   429,189          +9.28        +12.94           13,941                +3.36      True
      pg-93-paya-lebar             93 PAYA LEBAR WAY 300,000    329,968   363,015          +9.99        +21.01           33,047               +10.02      True
pg-241-jurong-east-s24     241 JURONG EAST STREET 24 468,000    407,134   424,711         -13.01         -9.25           17,577                +4.32      True
  pg-311c-clementi-av4        311C CLEMENTI AVENUE 4 675,000    646,679   615,785          -4.20         -8.77          -30,894                -4.78      True

/var/folders/t4/jqyfdvcn0xn_qjk47656g9840000gn/T/ipykernel_64113/3800571513.py:111: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  pred = float(predict_price(X.reshape(1, -1), bundle))


## How to read the comparison

- If **`mean_bias_pct`** for `03b (live)` is meaningfully more positive than `03 (table)`, the live-geo override is the main driver of the API's overestimation.
- If **`live_minus_table_pct`** is consistently positive across rows (not random noise), the live distances are systematically smaller than the table values — confirms the "town-median vs block-specific" hypothesis.
- If geocode fails for some rows (`geocoded == False`), those rows fall back to table values and should match `03` exactly — useful sanity check.